# Lab 3: Observability & Evaluation

In this lab, we will tackle the **Black Box Problem** by building a custom agent tracer, diagnosing infinite loops, and implementing circuit breakers. Finally, we will use **DeepEval** to evaluate our agent's performance.

## Setup
First, let's install the dependencies and configure our environment.

In [1]:
!pip install -q litellm python-dotenv deepeval

import os
from dotenv import load_dotenv

load_dotenv()

# Ensure you have your OPENAI_API_KEY set in your .env file
print("Environment loaded.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.18 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.6 which is incompatible.
mistralai 1.11.1 requires opentelemetry-semantic-conventions<0.60,>=0.59b0, but you have opentelemetry-semantic-conventions 0.62b0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.41.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.41.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
transformers 4.47.0 requires tokenizers<0.22,>=0.21, but you have tokenizers 0.22.2 which is

## Step 1: Instrumentation 🔬

Let's build an `AgentTracer` and inject it into a basic ReAct Agent. This will allow us to observe every step the agent takes.

In [2]:
from dataclasses import dataclass, field
import json
import time
from typing import Optional

@dataclass
class ToolCallRecord:
    tool_name: str
    tool_input: dict
    tool_output: str
    duration_ms: float

@dataclass
class AgentStep:
    step_number: int
    reasoning: Optional[str] = None
    tool_calls: list[ToolCallRecord] = field(default_factory=list)
    cost_usd: float = 0.0

@dataclass
class Trace:
    trace_id: str
    agent_name: str
    steps: list[AgentStep] = field(default_factory=list)
    status: str = "running"
    total_cost_usd: float = 0.0

class AgentTracer:
    def __init__(self):
        self.traces = {}

    def start_trace(self, trace_id, agent_name) -> str:
        t = Trace(trace_id=trace_id, agent_name=agent_name)
        self.traces[trace_id] = t
        return trace_id

    def log_step(self, trace_id: str, step: AgentStep):
        if trace_id in self.traces:
            self.traces[trace_id].steps.append(step)
            self.traces[trace_id].total_cost_usd += step.cost_usd

    def end_trace(self, trace_id: str, status="completed"):
        if trace_id in self.traces:
            self.traces[trace_id].status = status

    def print_trace(self, trace_id: str):
        t = self.traces.get(trace_id)
        if not t:
            print("Trace not found.")
            return
            
        print(f"=== TRACE SUMMARY: {t.trace_id} ===")
        print(f"Agent: {t.agent_name} | Status: {t.status}")
        print("-" * 30)
        for step in t.steps:
            print(f"  Step {step.step_number}: cost=${step.cost_usd:.4f}")
            for tc in step.tool_calls:
                print(f"    -> {tc.tool_name}({tc.tool_input}) [{tc.duration_ms:.0f}ms]")
        print("-" * 30)
        print(f"Total Cost: ${t.total_cost_usd:.4f}")
        print("=================================")

tracer = AgentTracer()


Now, let's define a broken tool and an agent that is prone to looping.

In [5]:
import litellm
import uuid

# A dummy tool that always returns the same confusing error
def search(query: str):
    time.sleep(0.5)
    return "Error: Database timeout. Please try again."

# Let's write a simplified Agent loop that incorporates the Tracer
def run_agent(query: str, tracer: AgentTracer, max_steps=5):
    trace_id = str(uuid.uuid4())[:8]
    tracer.start_trace(trace_id, "broken_react_agent")
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant. You must use the 'search' tool to answer queries. Format: search({\"query\": \"...\"})"},
        {"role": "user", "content": query}
    ]
    
    tools = [{
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the web for information.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"]
            }
        }
    }]
    
    for step_num in range(1, max_steps + 1):
        response = litellm.completion(
            model="openrouter/minimax/minimax-m2.5:free",
            messages=messages,
            tools=tools,
            temperature=0
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump())
        
        # Try calculating cost safely
        try:
            cost = litellm.completion_cost(completion_response=response) or 0.0
        except:
            cost = 0.0
            
        step_record = AgentStep(step_number=step_num, reasoning=msg.content, cost_usd=cost)
        
        if not hasattr(msg, 'tool_calls') or not msg.tool_calls:
            tracer.log_step(trace_id, step_record)
            tracer.end_trace(trace_id, "completed")
            return msg.content, trace_id
            
        for tc in msg.tool_calls:
            if tc.function.name == "search":
                args = json.loads(tc.function.arguments)
                start = time.time()
                result = search(args['query'])
                duration = (time.time() - start) * 1000
                
                step_record.tool_calls.append(
                    ToolCallRecord("search", args, result, duration)
                )
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
                
        tracer.log_step(trace_id, step_record)
        
    tracer.end_trace(trace_id, "loop_detected_max_steps")
    return "Reached max steps.", trace_id

# Run a query that triggers the loop
final_answer, tid = run_agent("What is the capital of France?", tracer)
tracer.print_trace(tid)



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



RateLimitError: litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Reset":"1776729600000"},"provider_name":null}},"user_id":"user_2zdpSBoUbNeZ8couMhUq6nCatsp"}

## Step 2 & 3: Diagnosis and The Fix 🔧

Notice how the step count repeated until `max_steps` and cost accumulated? Let's implement an **Exact Match Loop Detector** to act as a **Circuit Breaker**.

In [ ]:
class LoopDetector:
    def __init__(self):
        self.history = []
        
    def check_tool_call(self, name: str, args: str) -> bool:
        """Returns True if this is a loop."""
        call = (name, args)
        count = self.history.count(call)
        self.history.append(call)
        
        # If we've seen this exact call 2 times before, it's a loop.
        if count >= 2:
            return True
        return False
        
detector = LoopDetector()

# Try checking some calls
print("Is loop?", detector.check_tool_call("search", '{"query": "France"}'))
print("Is loop?", detector.check_tool_call("search", '{"query": "France"}'))
print("Is loop?", detector.check_tool_call("search", '{"query": "France"}')) # This one should trigger!


## Step 4: Verify with DeepEval ✅

DeepEval provides a production-grade evaluation framework. Let's create a complete evaluation suite for our agent, testing both **Task Completion** and **Tool Correctness** across multiple test cases.

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import TaskCompletionMetric, ToolCorrectnessMetric

# 1. Define a suite of test cases
test_cases = [
    LLMTestCase(
        name="Factual Query",
        input="What is the capital of France?",
        actual_output="Paris is the capital of France.",
        expected_output="Paris",
        tools_called=[ToolCall(name="search", input_parameters={"query": "capital of France"})]
    ),
    LLMTestCase(
        name="Math Query without Tool",
        input="What is 2 + 2?",
        actual_output="4",
        expected_output="4",
        tools_called=[]  # We expect no tools to be needed
    ),
    LLMTestCase(
        name="Failed Execution",
        input="Explain quantum string theory.",
        actual_output="I don't know anything about that.",
        expected_output="A detailed explanation of quantum string theory.",
        tools_called=[ToolCall(name="search", input_parameters={"query": "quantum string theory"})]
    )
]

# 2. Initialize our metrics
task_metric = TaskCompletionMetric(threshold=0.7)
tool_metric = ToolCorrectnessMetric(threshold=0.7)

# 3. Run the evaluation
# NOTE: Inside a jupyter notebook, we run evaluate directly.
# This will use litellm & your OPENAI_API_KEY as the LLM Judge.
print("Starting Evaluation Suite...")
evaluation_results = evaluate(
    test_cases=test_cases,
    metrics=[task_metric, tool_metric],
    print_results=False # We will print a custom dashboard below
)

Once the evaluation completes, we can iterate over the results to build our own reporting dashboard.

In [ ]:
print("======================================================================")
print("DEEPEVAL EVALUATION RESULTS")
print("======================================================================")
print(f"{str('Test Case').ljust(30)} {'Task Comp'.center(12)} {'Tool Correct'.center(12)}")
print("-" * 60)

passed = 0
for idx, result in enumerate(evaluation_results):
    name = test_cases[idx].name if hasattr(test_cases[idx], 'name') else f"Test {idx+1}"
    
    # Get metric success status
    task_pass = "PASS" if result.metrics_data[0].success else "FAIL"
    tool_pass = "PASS" if result.metrics_data[1].success else "FAIL"
    
    if result.success:
        passed += 1
        
    print(f"{name[:28].ljust(30)} {task_pass.center(12)} {tool_pass.center(12)}")

print("-" * 60)
print(f"PASS RATE: {int((passed/len(test_cases))*100)}% ({passed}/{len(test_cases)})")
print("======================================================================")


## Step 5: Replacing Manual Tracing with `@observe` ✨

In **Step 1**, we wrote an `AgentTracer` from scratch. We had to manually call `tracer.start_trace()`, log steps, and measure durations.

DeepEval's `@observe` decorator entirely replaces this boilerplate by automatically intercepting your functions and building the trace graph (Trace -> Root Span -> Child Spans) for you. Let's refactor our agent to use it!

In [ ]:
from deepeval.tracing import observe
import litellm
import json

class ObservableReactAgent:
    
    @observe(type="tool")
    def execute_search(self, query: str):
        # DeepEval automatically records the inputs and duration of this tool span
        print(f"  [Tool] Searching for: {query}")
        return "Paris is the capital of France."

    @observe(type="llm")
    def call_llm(self, messages, tools):
        # This creates an LLM span inside our trace graph
        return litellm.completion(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            temperature=0
        )
        
    @observe(type="agent")
    def run(self, query: str):
        # This creates the Root Span for the entire trace
        print(f"[Agent] Started trace for: {query}")
        
        messages = [{"role": "user", "content": query}]
        tools = [{
            "type": "function",
            "function": {
                "name": "execute_search",
                "parameters": {"type": "object", "properties": {"query": {"type": "string"}}}
            }
        }]
        
        # 1. Call LLM
        response = self.call_llm(messages, tools)
        msg = response.choices[0].message
        
        # 2. Execute Tool if requested
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                tool_result = self.execute_search(args['query'])
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": tool_result})
                
            # 3. Call LLM again with results
            final_response = self.call_llm(messages, tools)
            return final_response.choices[0].message.content
            
        return msg.content

# Run our observable agent!
# Note: In a real environment, @observe will automatically transmit traces 
# to Confident AI or another tracing backend you configure.
agent = ObservableReactAgent()
final_answer = agent.run("What is the capital of France?")
print("Final Answer:", final_answer)


### Why is `@observe` better?

1. **Zero Boilerplate**: No need to instantiate an `AgentTracer` or manually calculate `time.time()` durations.
2. **Automatic Hierarchy**: Decorating the class methods natively builds the nested parent-child tree (Agent -> LLM / Tool).
3. **Metrics Integration**: You can attach metrics directly to the decorator (e.g. `@observe(metrics=[TaskCompletionMetric()])`), completely unifying Tracing and Evaluation!